# P0: full index set vs edge-type counts (E3 archived outputs, no training)

Re-runs `docs/research/scripts/p0_full_vs_edgetypes.py` on the **public** E3 dataset and checks the result
against the CSVs committed in `docs/research/out/P0/`.

* Runtime: CPU only, about 5–10 min (most of it is downloading and aggregating OPERA in step 3).
* Nothing is trained. The contrast compares the *selected* model for each representation, so it does **not**
  remove learner confounding (extra trees vs SVR); see `docs/research/14_FIXED_LEARNER_PROTOCOL.md`.
* The OPERA/PHYSPROP measured values are not redistributed in the repo (upstream terms unknown). Step 3
  downloads the pinned public OPERA zip and restores them, then checks them against recorded sha256 hashes.* If Colab says the session must be restarted after step 1 (because numpy was replaced), restart it and run all
  cells again from the top. The installs are then no-ops.
* Expected final output: `max |difference| = 0.00e+00 OK` for both files. A `FAIL` in step 3 means the
  restored OPERA values do not match the recorded hashes. Do not use the run in that case.


In [ ]:
BRANCH = "research-2026-10-13"
!git clone -q --depth 1 --branch $BRANCH https://github.com/Singati2/TopoQSPR.git /content/TopoQSPR
%cd /content/TopoQSPR
!git log --oneline -1

In [ ]:
# 1. Install, pinned to the versions that produced the committed results (Python 3.10 locally; Colab's
#    newer Python is fine). RDKit and chembl_structure_pipeline matter most: they decide the InChIKeys
#    used to re-attach the OPERA values in step 3.
!pip install -q "rdkit==2026.3.6" "chembl_structure_pipeline==1.2.4" "numpy==2.2.6" "pandas==2.3.3" \
    "scipy==1.15.3" "scikit-learn==1.7.2" "openpyxl==3.1.5" "joblib==1.6.0" "xgboost==3.2.0"
!pip install -q --no-deps -e .
import rdkit, sklearn, pandas, numpy, scipy
print("rdkit", rdkit.__version__, "| sklearn", sklearn.__version__, "| pandas", pandas.__version__,
      "| numpy", numpy.__version__, "| scipy", scipy.__version__)

In [ ]:
# 2. Verify every published dataset file against SHA256SUMS
import hashlib, pathlib
root = pathlib.Path("docs/research/data/E3")
bad = []
for line in (root / "SHA256SUMS").read_text().splitlines():
    h, rel = line.split("  ", 1)
    if hashlib.sha256((root / rel).read_bytes()).hexdigest() != h:
        bad.append(rel)
print("all", len((root / "SHA256SUMS").read_text().splitlines()), "files match" if not bad else f"MISMATCH: {bad}")
assert not bad

In [ ]:
# 3. (about 1-3 min; downloads ~80 MB) Restore the OPERA measured values (downloads the pinned OPERA zip, sha256-checked) and verify them
!python -I docs/research/scripts/restore_opera_values.py docs/research/data/E3 /content/e3_restored

In [ ]:
# 4. Run P0 on the restored dataset
!python -I docs/research/scripts/p0_full_vs_edgetypes.py /content/e3_restored /content/p0_out

In [ ]:
# 5. Compare with the committed results
import pandas as pd, numpy as np
for name in ["internal_full_vs_edgetypes.csv", "external_full_vs_edgetypes.csv"]:
    a = pd.read_csv(f"docs/research/out/P0/{name}"); b = pd.read_csv(f"/content/p0_out/{name}")
    num = a.select_dtypes("number").columns
    diff = np.nanmax(np.abs(a[num].to_numpy(float) - b[num].to_numpy(float)))
    print(f"{name}: max |difference| = {diff:.2e}", "OK" if diff < 1e-9 else "CHECK")